# Mission 3 - Colab 단일 노트북

이 파일 **하나만 Colab에 올린 뒤 위에서부터 순서대로 실행**하면 됩니다.

이 노트북이 하는 일:

1. 필요 패키지 설치
2. Google Drive 연결
3. `/content`에 올린 Training/Validation JSON 폴더 자동 탐색
4. 대화 `text`만으로 TF-IDF + Logistic Regression 학습
5. Training 내부 분할 Macro F1 확인
6. Training 전체로 최종 모델 학습
7. Validation 예측 CSV와 Macro F1 생성
8. 모델, CSV, 점수를 Google Drive에 영구 저장

## 출제 규칙

- 학습은 서울 Training만 사용합니다.
- 모델 입력은 JSON의 `utterances[].text`만 사용합니다.
- `symptom`은 학습 정답 또는 최종 평가 정답으로만 사용합니다.
- Validation은 학습과 임계값 조정에 사용하지 않습니다.
- 9개 외 symptom만 제외하고 통화 샘플은 제거하지 않습니다.

In [ ]:
# 1. 필요 패키지 설치
import importlib.util
import subprocess
import sys

required = {
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("설치 완료:", missing)
else:
    print("필요 패키지가 이미 설치되어 있습니다.")

In [ ]:
# 2. Google Drive 연결 - 모델과 CSV만 Drive에 영구 저장합니다.
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    OUTPUT_DIR = Path("/content/drive/MyDrive/DCC_AIdeas/mission3_outputs")
except ImportError:
    # Colab 외의 환경에서 실행할 때의 예비 경로입니다.
    OUTPUT_DIR = Path.cwd() / "mission3_outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "mission3_model.ckpt"
PREDICTION_PATH = OUTPUT_DIR / "mission3.csv"
METRICS_PATH = OUTPUT_DIR / "mission3_metrics.json"

print("영구 저장 폴더:", OUTPUT_DIR)
print("모델 저장 경로:", CHECKPOINT_PATH)

## 3. Google Drive 또는 `/content`에 있는 데이터 찾기

아래 셀은 연결된 Google Drive의 `MyDrive`를 먼저 확인하고, Drive를 사용할 수 없으면 `/content`에서 `TL_서울_구급`과 `VL_서울_구급` 폴더를 자동으로 찾습니다.

데이터 폴더 이름을 바꾸지 않았다면 경로를 직접 입력할 필요가 없습니다.

In [ ]:
# 3. Training/Validation 라벨 폴더 자동 탐색
import os

DRIVE_ROOT = Path("/content/drive/MyDrive")
CONTENT_ROOT = DRIVE_ROOT if DRIVE_ROOT.is_dir() else (
    Path("/content") if Path("/content").is_dir() else Path.cwd()
)
SKIP_DIRS = {"drive", "sample_data", ".config", "__pycache__", ".git"}
TARGET_FOLDER_NAMES = {"TL_서울_구급", "VL_서울_구급"}

# 자주 사용하는 경로를 먼저 확인하여 전체 파일 탐색을 피합니다.
common_parents = [
    CONTENT_ROOT / "mission3_data",
    CONTENT_ROOT / "대학부 데이터",
    CONTENT_ROOT / "DATA" / "대학부 데이터",
    CONTENT_ROOT / "DCC_AIdeas" / "DATA" / "대학부 데이터",
]
direct_candidates = {
    "TL_서울_구급": [
        CONTENT_ROOT / "mission3_data" / "train" / "TL_서울_구급",
        *[parent / "Training" / "2.라벨링데이터" / "TL_서울_구급" for parent in common_parents],
    ],
    "VL_서울_구급": [
        CONTENT_ROOT / "mission3_data" / "validation" / "VL_서울_구급",
        *[parent / "Validation" / "2.라벨링데이터" / "VL_서울_구급" for parent in common_parents],
    ],
}

located = {}
for folder_name, candidates in direct_candidates.items():
    located[folder_name] = next((path for path in candidates if path.is_dir()), None)

missing_names = {name for name, path in located.items() if path is None}
if missing_names:
    print("일반 경로에서 못 찾은 폴더만 한 번 추가 탐색합니다:", missing_names)
    for current, directories, _ in os.walk(CONTENT_ROOT):
        directories[:] = [name for name in directories if name not in SKIP_DIRS]
        current_path = Path(current)
        if current_path.name in missing_names:
            located[current_path.name] = current_path
            directories[:] = []
            missing_names.remove(current_path.name)
            if not missing_names:
                break

if missing_names:
    raise FileNotFoundError(
        f"{sorted(missing_names)} 폴더를 {CONTENT_ROOT}에서 찾지 못했습니다. "
        "압축을 풀었는지와 폴더 이름을 확인하세요."
    )

TRAIN_LABEL_DIR = located["TL_서울_구급"]
VALID_LABEL_DIR = located["VL_서울_구급"]

print("Training JSON 폴더:", TRAIN_LABEL_DIR)
print("Validation JSON 폴더:", VALID_LABEL_DIR)
print("Training JSON 개수:", len(list(TRAIN_LABEL_DIR.rglob("*.json"))))
print("Validation JSON 개수:", len(list(VALID_LABEL_DIR.rglob("*.json"))))

In [ ]:
# 4. Mission 3 데이터 처리 함수 - 모델 입력으로는 text만 사용합니다.
import csv
import json
from collections import Counter
from typing import Any

import joblib
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier

TARGET_SYMPTOMS = (
    "고열", "구토", "두통", "복통", "어지러움",
    "열상", "오심", "전신쇠약", "호흡곤란",
)
UTTERANCE_KEYS = ("utterances", "utterences")

def find_label_files(label_dir: Path) -> list[Path]:
    files = sorted(label_dir.rglob("*.json"))
    if not files:
        raise ValueError(f"JSON 파일이 없습니다: {label_dir}")
    return files

def load_annotation(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as file:
        annotation = json.load(file)
    if not isinstance(annotation, dict):
        raise ValueError(f"JSON object가 아닙니다: {path}")
    return annotation

def dialogue_text(annotation: dict[str, Any]) -> str:
    utterances = []
    for key in UTTERANCE_KEYS:
        if isinstance(annotation.get(key), list):
            utterances = annotation[key]
            break
    texts = []
    for utterance in utterances:
        if not isinstance(utterance, dict):
            continue
        text = utterance.get("text")
        if isinstance(text, str) and text.strip():
            texts.append(text.strip())
    return "\n".join(texts)

def target_symptoms(annotation: dict[str, Any]) -> list[str]:
    raw = annotation.get("symptom", [])
    if raw is None:
        raw = []
    elif isinstance(raw, str):
        raw = [raw]
    if not isinstance(raw, list):
        raise ValueError("symptom은 문자열 list여야 합니다.")
    found = {str(value).strip() for value in raw}
    return [symptom for symptom in TARGET_SYMPTOMS if symptom in found]

def target_vector(annotation: dict[str, Any]) -> list[int]:
    found = set(target_symptoms(annotation))
    return [int(symptom in found) for symptom in TARGET_SYMPTOMS]

def load_dataset(label_dir: Path, include_targets: bool):
    paths = find_label_files(label_dir)
    texts, targets = [], []
    for index, path in enumerate(paths, 1):
        annotation = load_annotation(path)
        texts.append(dialogue_text(annotation))
        if include_targets:
            targets.append(target_vector(annotation))
        if index % 5000 == 0:
            print(f"  {index:,}/{len(paths):,} JSON 읽음")
    target_array = np.asarray(targets, dtype=np.int8) if include_targets else None
    return paths, texts, target_array

print("데이터 처리 함수 준비 완료")

In [ ]:
# 5. Training 데이터를 읽고 9개 증상 분포를 확인합니다.
training_paths, training_texts, training_targets = load_dataset(
    TRAIN_LABEL_DIR, include_targets=True
)

print("\nTraining 샘플:", len(training_paths))
print("빈 대화 텍스트:", sum(not text.strip() for text in training_texts))
print("증상별 정답 개수:")
for symptom, count in zip(TARGET_SYMPTOMS, training_targets.sum(axis=0)):
    print(f"  {symptom:<8} {int(count):>6,}")

print("\n첫 대화 미리보기:")
print(training_texts[0][:500])

## 6. 학습 설정

첫 실험에서는 아래 기본값을 그대로 사용하세요. 메모리가 부족하면 `MAX_FEATURES` 값만 `50000`으로 낮추면 됩니다.

In [ ]:
# 6. 학습 설정과 모델 함수
SEED = 42
TEST_SIZE = 0.20
MAX_FEATURES = 100_000
MIN_DF = 2
LOGISTIC_C = 2.0
MAX_ITER = 1_000
N_JOBS = 1  # 메모리 폭증을 피하기 위해 안전하게 1로 시작합니다.

def make_vectorizer() -> TfidfVectorizer:
    return TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2, 5),
        min_df=MIN_DF,
        max_features=MAX_FEATURES,
        sublinear_tf=True,
        dtype=np.float32,
    )

def make_classifier() -> OneVsRestClassifier:
    estimator = LogisticRegression(
        C=LOGISTIC_C,
        class_weight="balanced",
        max_iter=MAX_ITER,
        penalty="l2",
        solver="liblinear",
        random_state=SEED,
    )
    return OneVsRestClassifier(estimator, n_jobs=N_JOBS)

def tune_thresholds(truth: np.ndarray, probabilities: np.ndarray) -> np.ndarray:
    thresholds = np.full(len(TARGET_SYMPTOMS), 0.5, dtype=np.float32)
    candidates = np.arange(0.10, 0.91, 0.05)
    for column in range(len(TARGET_SYMPTOMS)):
        if truth[:, column].min() == truth[:, column].max():
            continue
        best_score = -1.0
        for threshold in candidates:
            predicted = (probabilities[:, column] >= threshold).astype(np.int8)
            score = f1_score(truth[:, column], predicted, zero_division=0)
            if score > best_score:
                best_score = score
                thresholds[column] = threshold
    return thresholds

print("MAX_FEATURES:", MAX_FEATURES)
print("N_JOBS:", N_JOBS)

In [ ]:
# 7. Training 내부 80/20 분할로 점수를 구한 뒤 Training 전체로 최종 모델을 학습합니다.
train_texts, holdout_texts, train_targets, holdout_targets = train_test_split(
    training_texts,
    training_targets,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True,
)

print("1/4 내부 확인용 TF-IDF 학습...")
holdout_vectorizer = make_vectorizer()
train_features = holdout_vectorizer.fit_transform(train_texts)
holdout_features = holdout_vectorizer.transform(holdout_texts)

print("2/4 내부 확인용 분류기 학습...")
holdout_classifier = make_classifier()
holdout_classifier.fit(train_features, train_targets)
holdout_probabilities = holdout_classifier.predict_proba(holdout_features)
thresholds = tune_thresholds(holdout_targets, holdout_probabilities)
holdout_predictions = (holdout_probabilities >= thresholds).astype(np.int8)
holdout_scores = f1_score(
    holdout_targets, holdout_predictions, average=None, zero_division=0
)

print("\nTraining 내부 holdout 결과")
print("-" * 55)
for symptom, score, threshold in zip(TARGET_SYMPTOMS, holdout_scores, thresholds):
    print(f"{symptom:<8} F1={score:.4f}  threshold={threshold:.2f}")
print("-" * 55)
print(f"Macro F1 = {holdout_scores.mean():.4f}\n")

print("3/4 Training 전체 TF-IDF 학습...")
final_vectorizer = make_vectorizer()
all_features = final_vectorizer.fit_transform(training_texts)

print("4/4 Training 전체 분류기 학습...")
final_classifier = make_classifier()
final_classifier.fit(all_features, training_targets)

checkpoint = {
    "vectorizer": final_vectorizer,
    "classifier": final_classifier,
    "thresholds": thresholds,
    "target_symptoms": TARGET_SYMPTOMS,
    "model_name": "char TF-IDF + One-vs-Rest Logistic Regression",
    "training_samples": len(training_texts),
    "seed": SEED,
}
joblib.dump(checkpoint, CHECKPOINT_PATH)
print("최종 모델 저장 완료:", CHECKPOINT_PATH)

In [ ]:
# 8. Validation에서는 오직 text만 읽어 예측 CSV를 만듭니다.
validation_paths, validation_texts, _ = load_dataset(
    VALID_LABEL_DIR, include_targets=False
)

saved_checkpoint = joblib.load(CHECKPOINT_PATH)
validation_features = saved_checkpoint["vectorizer"].transform(validation_texts)
validation_probabilities = saved_checkpoint["classifier"].predict_proba(
    validation_features
)
saved_thresholds = np.asarray(saved_checkpoint["thresholds"], dtype=np.float32)
validation_predictions = validation_probabilities >= saved_thresholds

rows = []
for path, flags in zip(validation_paths, validation_predictions):
    symptoms = [
        symptom for symptom, present in zip(TARGET_SYMPTOMS, flags) if present
    ]
    rows.append((path.name, str(symptoms)))

with PREDICTION_PATH.open("w", newline="", encoding="utf-8-sig") as file:
    writer = csv.writer(file)
    writer.writerow(["label file name", "symptom"])
    writer.writerows(rows)

print(f"{len(rows):,}개 예측 CSV 저장 완료: {PREDICTION_PATH}")
print("예측 예시:", rows[:5])

In [ ]:
# 9. Validation은 학습에 넣지 않고 여기서 최종 Macro F1만 계산합니다.
_, _, validation_targets = load_dataset(VALID_LABEL_DIR, include_targets=True)
validation_scores = f1_score(
    validation_targets,
    validation_predictions.astype(np.int8),
    average=None,
    zero_division=0,
)
validation_macro_f1 = float(validation_scores.mean())

print("Validation 결과")
print("-" * 40)
for symptom, score in zip(TARGET_SYMPTOMS, validation_scores):
    print(f"{symptom:<8} F1={score:.4f}")
print("-" * 40)
print(f"Validation Macro F1 = {validation_macro_f1:.4f}")

metrics = {
    "model": saved_checkpoint["model_name"],
    "training_samples": len(training_texts),
    "validation_samples": len(validation_texts),
    "training_holdout_macro_f1": float(holdout_scores.mean()),
    "validation_macro_f1": validation_macro_f1,
    "per_symptom_f1": {
        symptom: float(score) for symptom, score in zip(TARGET_SYMPTOMS, validation_scores)
    },
    "thresholds": {
        symptom: float(value) for symptom, value in zip(TARGET_SYMPTOMS, saved_thresholds)
    },
}
with METRICS_PATH.open("w", encoding="utf-8") as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2)
print("점수 기록 저장 완료:", METRICS_PATH)

## 10. 결과 파일

모든 결과는 Google Drive의 다음 폴더에 저장됩니다.

```text
MyDrive/DCC_AIdeas/mission3_outputs/
├─ mission3_model.ckpt
├─ mission3.csv
└─ mission3_metrics.json
```

아래 셀의 `DOWNLOAD_CSV`를 `True`로 바꾸면 CSV를 PC로도 다운로드합니다.

In [ ]:
# 10. 원할 때만 True로 바꾸어 CSV를 PC로 다운로드하세요.
DOWNLOAD_CSV = False

if DOWNLOAD_CSV:
    try:
        from google.colab import files
        files.download(str(PREDICTION_PATH))
    except ImportError:
        print("Colab에서만 자동 다운로드할 수 있습니다.")
else:
    print("CSV는 Drive에 저장되었습니다:", PREDICTION_PATH)

## 11. 실험 메모

실행이 끝나면 아래를 채우고 **노트북을 저장**하세요. 셀 출력이 학습 로그가 됩니다.

- 실험 날짜:
- Training 내부 Macro F1:
- Validation Macro F1:
- 가장 잘 맞힌 증상:
- 가장 못 맞힌 증상:
- 다음 실험에서 하나만 바꿀 설정:

> 이 노트북의 임계값은 Training 내부 holdout에서만 조정합니다. Validation 점수를 보며 계속 임계값을 바꾸면 Validation을 학습에 사용한 것과 비슷해지므로 피하세요.